# Stroke Prediction

## Importing necessary libraries

In [1]:
import numpy as np
import pandas as pd 

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from imblearn.under_sampling import TomekLinks
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, fbeta_score, make_scorer

## Import dataset

In [2]:
df = pd.read_csv('../dataset/healthcare-dataset-stroke-data.csv')

In [3]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [5]:
df.describe()

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


In [6]:
df.describe(include=[object])

,gender,ever_married,work_type,Residence_type,smoking_status
count,5110,5110,5110,5110,5110
unique,3,2,5,2,4
top,Female,Yes,Private,Urban,never smoked
freq,2994,3353,2925,2596,1892


## Rename column for consistent formatting of name

In [7]:
df.rename(columns={'Residence_type': 'residence_type'}, inplace=True)

In [8]:
pd.set_option('display.max_columns', None)

In [9]:
pd.set_option('display.max_rows', None)

### Impute missing values in BMI column
For 201 entries missing BMI values, these entries will be imputed with the median BMI value

In [10]:
median_bmi = df['bmi'].median()
df['bmi'].fillna(median_bmi, inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7112\2337012406.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['bmi'].fillna(median_bmi, inplace=True)


In [11]:
print(df['bmi'].isna().sum())

0


## Feature Engineering

### Converting categorical columns to numerical columns

In [12]:
df['male'] = (df['gender'] == 'Male').astype(int)

In [13]:
df['ever_married'] = (df['ever_married'] == 'Yes').astype(int)

In [14]:
df['urban'] = (df['residence_type'] == 'Urban').astype(int)

In [15]:
df['smoke'] = ((df['smoking_status'] == 'formerly smoked') | (df['smoking_status'] == 'smokes')).astype(int)

In [16]:
df['have_work'] = ((df['work_type'] == 'Govt_job') | (df['work_type'] == 'Private') | (df['work_type'] == 'Self-employed')).astype(int)

In [17]:
df['hypertension_or_heart_disease'] = df['hypertension'] | df['heart_disease']

In [18]:
df = pd.get_dummies(df, columns=['work_type', 'smoking_status'], drop_first = False)

### Extra column for the age of married people

In [19]:
df['age_ever_married'] = df['age'] * df['ever_married']

In [20]:
df['age_hypertension'] = df['age'] * df['hypertension']

In [21]:
df['age_heart_disease'] = df['age'] * df['heart_disease']

In [22]:
df['age_bmi'] = df['age'] * df['bmi']

In [23]:
df['age_avg_glucose_level'] = df['age'] * df['avg_glucose_level']

In [24]:
df['bmi_avg_glucose_level'] = df['bmi'] * df['avg_glucose_level']

In [25]:
df['avg_glucose_level_hypertension'] = df['avg_glucose_level'] * df['hypertension']

### Creating columns representing outstanding values for BMI and Average Glucose Level

In [26]:
df['obese'] = df['bmi'] >= 30

In [27]:
df['high_avg_glucose_level'] = df['avg_glucose_level'] >= 150

In [28]:
df['obese_high_avg_glucose_level'] = df['obese'] | df['high_avg_glucose_level']

In [29]:
df['old'] = df['age'] >= 65

### Converting boolean columns to numerical

In [30]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

In [31]:
df['risk_count'] = df['obese'] + df['high_avg_glucose_level'] + df['old'] + df['hypertension'] + df['heart_disease']

### Reformatting column names

In [32]:
df.rename(columns={'work_type_Govt_job': 'work_type_govt_job', 'work_type_Never_worked': 'work_type_never_worked', 'work_type_Private': 'work_type_private', 'work_type_Self-employed': 'work_type_self_employed'}, inplace=True)

In [33]:
df.rename(columns={'smoking_status_Unknown': 'smoking_status_unknown', 'smoking_status_formerly smoked': 'smoking_status_formerly_smoked', 'smoking_status_never smoked': 'smoking_status_never_smoked'}, inplace=True)

In [34]:
df.drop(columns=['gender', 'residence_type'], inplace=True)

### Log Transforming BMI and Average Glucose Level columns

In [35]:
df['log_bmi'] = np.log(df['bmi'])

In [36]:
df['log_avg_glucose_level'] = np.log(df['avg_glucose_level'])

In [37]:
df.head()

,id,age,hypertension,heart_disease,ever_married,avg_glucose_level,bmi,stroke,male,urban,smoke,have_work,hypertension_or_heart_disease,work_type_govt_job,work_type_never_worked,work_type_private,work_type_self_employed,work_type_children,smoking_status_unknown,smoking_status_formerly_smoked,smoking_status_never_smoked,smoking_status_smokes,age_ever_married,age_hypertension,age_heart_disease,age_bmi,age_avg_glucose_level,bmi_avg_glucose_level,avg_glucose_level_hypertension,obese,high_avg_glucose_level,obese_high_avg_glucose_level,old,risk_count,log_bmi,log_avg_glucose_level
0,9046,67.0,0,1,1,228.69,36.6,1,1,1,1,1,1,0,0,1,0,0,0,1,0,0,67.0,0.0,67.0,2452.2,15322.23,8370.054,0.00,1,1,1,1,4,3.600048,5.432367
1,51676,61.0,0,0,1,202.21,28.1,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,61.0,0.0,0.0,1714.1,12334.81,5682.101,0.00,0,1,1,0,1,3.335770,5.309307
2,31112,80.0,0,1,1,105.92,32.5,1,1,0,0,1,1,0,0,1,0,0,0,0,1,0,80.0,0.0,80.0,2600.0,8473.60,3442.400,0.00,1,0,1,1,3,3.481240,4.662684
3,60182,49.0,0,0,1,171.23,34.4,1,0,1,1,1,0,0,0,1,0,0,0,0,0,1,49.0,0.0,0.0,1685.6,8390.27,5890.312,0.00,1,1,1,0,2,3.538057,5.143008
4,1665,79.0,1,0,1,174.12,24.0,1,0,0,0,1,1,0,0,0,1,0,0,0,1,0,79.0,79.0,0.0,1896.0,13755.48,4178.880,174.12,0,1,1,1,3,3.178054,5.159745


In [38]:
df.shape

(5110, 36)

## Model Building

### Splitting to training, validation, and testing sets

In [39]:
columns_name = columns_name = [col for col in df.columns if col != 'id']

In [40]:
X = df[columns_name]
X.drop('stroke', axis=1, inplace=True)
y = df['stroke']

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7112\91966498.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.drop('stroke', axis=1, inplace=True)


In [41]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5, random_state = 42, stratify = y_test)

### ADASYN Oversampling on Training Data

In [42]:
resampler = TomekLinks()

In [43]:
X_train_res, y_train_res = resampler.fit_resample(X_train, y_train)

### Create base random forest classifier model

In [44]:
base_classifier = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

### Create grid for hyperparameter tuning

In [45]:
param_grid_xgb = {
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'scale_pos_weight': [1, 3, 5, 10, 15, 20]
}

In [46]:
f2_scorer = make_scorer(fbeta_score, beta=2)

In [47]:
grid_xgb = GridSearchCV(
    estimator=base_classifier,
    param_grid=param_grid_xgb,
    scoring=f2_scorer,
    cv=5,
    verbose=2,
    n_jobs=-1
)

In [48]:
# importance_df = pd.DataFrame({'Feature': X_train_res.columns, 'Importance': base_classifier.feature_importances_})
# importance_df = importance_df.sort_values(by='Importance', ascending=False)
# print(importance_df)

### Fitting base classifier on unsampled data

In [49]:
base_classifier.fit(X_train_res, y_train_res)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [50]:
importances = base_classifier.feature_importances_

In [51]:
feature_imp_df = pd.DataFrame({'Feature':[col for col in X_train], 'Importance': importances}).sort_values(
    'Importance', ascending=False)

In [52]:
print(feature_imp_df)

                           Feature  Importance
0                              age    0.100594
2                    heart_disease    0.092366
3                     ever_married    0.074564
23                         age_bmi    0.058238
10   hypertension_or_heart_disease    0.046116
21                age_hypertension    0.045385
19           smoking_status_smokes    0.043820
20                age_ever_married    0.042206
11              work_type_govt_job    0.042125
18     smoking_status_never_smoked    0.038685
5                              bmi    0.037237
14         work_type_self_employed    0.035073
4                avg_glucose_level    0.031637
29    obese_high_avg_glucose_level    0.031068
31                      risk_count    0.028246
17  smoking_status_formerly_smoked    0.027801
24           age_avg_glucose_level    0.027561
16          smoking_status_unknown    0.027519
25           bmi_avg_glucose_level    0.026254
26  avg_glucose_level_hypertension    0.025215
6            

In [53]:
# retained_columns = ['age', 'heart_disease', 'ever_married', 'age_bmi', 'hypertension_or_heart_disease', 'age_hypertension', 'smoking_status_smokes', 'age_ever_married', 'work_type_govt_job']

In [54]:
retained_columns = ['age', 'heart_disease', 'ever_married', 'age_bmi', 'hypertension_or_heart_disease', 'age_hypertension', 'smoking_status_smokes', 'age_ever_married', 'work_type_govt_job', 'smoking_status_never_smoked']

In [55]:
# retained_columns = ['age', 'hypertension_or_heart_disease', 'work_type_govt_job', 'smoking_status_formerly_smoked', 'smoking_status_unknown', 'bmi', 'work_type_self_employed', 'smoking_status_smokes', 'avg_glucose_level', 'smoking_status_never_smoked']

In [56]:
X_train_res_retained = X_train_res[retained_columns]

In [57]:
X_val_retained = X_val[retained_columns]

### Create optimizing function for hyperparameter tuning with Optuna

In [58]:
MIN_PRECISION = 0.2

In [59]:
def objective_xgb(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.4, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 20),
        'tree_method': trial.suggest_categorical('tree_method', ['hist', 'approx']), 
        'grow_policy': trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),  
        'max_leaves': trial.suggest_int('max_leaves', 0, 256),
        'min_split_loss': trial.suggest_float('min_split_loss', 0, 5),  
        'max_bin': trial.suggest_int('max_bin', 128, 512),
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    model = xgb.XGBClassifier(**params)
    
    model.fit(X_train_res_retained, y_train_res)
    
    y_pred_val = model.predict(X_val_retained)
    
    recall = recall_score(y_val, y_pred_val)
    precision = precision_score(y_val, y_pred_val)
    f2_score = fbeta_score(y_val, y_pred_val, beta=2)
    
    if precision < MIN_PRECISION:
        return 0
    # return recall + (precision * 0.01)
    return f2_score + recall * 0.001 + precision * 0.00001

In [60]:
# grid_xgb.fit(X_train_res_retained, y_train_res)

In [61]:
study_xgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=500, show_progress_bar=True)

[I 2026-01-22 14:57:35,841] A new study created in memory with name: no-name-66511d04-6672-498e-a079-2a1b73b11168


  0%|          | 0/500 [00:00<?, ?it/s]

[I 2026-01-22 14:57:37,141] Trial 0 finished with value: 0.0 and parameters: {'max_depth': 7, 'learning_rate': 0.24517932047070642, 'n_estimators': 380, 'min_child_weight': 6, 'max_delta_step': 1, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.4348501673009197, 'colsample_bylevel': 0.9197056874649611, 'colsample_bynode': 0.7606690070459252, 'gamma': 3.540362888980227, 'reg_alpha': 0.20584494295802447, 'reg_lambda': 9.699098521619943, 'scale_pos_weight': 16.816410175208013, 'tree_method': 'hist', 'grow_policy': 'lossguide', 'max_leaves': 134, 'min_split_loss': 2.1597250932105787, 'max_bin': 240}. Best is trial 0 with value: 0.0.


c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[I 2026-01-22 14:57:38,174] Trial 1 finished with value: 0.0 and parameters: {'max_depth': 10, 'learning_rate': 0.008851384099881301, 'n_estimators': 181, 'min_child_weight': 4, 'max_delta_step': 5, 'subsample': 0.8925879806965068, 'colsample_bytree': 0.5198042692950159, 'colsample_bylevel': 0.708540663048167, 'colsample_bynode': 0.7554487413172255, 'gamma': 0.23225206359998862, 'reg_alpha': 6.075448519014383, 'reg_lambda': 1.7052412368729153, 'scale_pos_weight': 2.235980266720311, 'tree_method': 'approx', 'grow_policy': 'depthwise', 'max_leaves': 25, 'min_split_loss': 3.4211651325607844, 'max_bin': 297}. Best is trial 0 with value: 0.0.
[I 2026-01-22 14:57:38,337] Trial 2 finished with value: 0.0 and parameters: {'max_depth': 4, 'learning_rate': 0.03797252233376349, 'n_estimators': 65, 'min_child_weight': 10, 'max_delta_step': 2, 'subsample': 0.831261142176991, 'colsample_bytree': 0.5870266456536466, 'colsample_bylevel': 0.7120408127066865, 'colsample_bynode': 0.7280261676059678, 'gam

...

[I 2026-01-22 14:58:34,070] Trial 35 finished with value: 0.0 and parameters: {'max_depth': 6, 'learning_rate': 0.0926311118027599, 'n_estimators': 338, 'min_child_weight': 10, 'max_delta_step': 6, 'subsample': 0.5274760681339411, 'colsample_bytree': 0.7859982816814315, 'colsample_bylevel': 0.6534024461475244, 'colsample_bynode': 0.6951534103058257, 'gamma': 2.3168043775739218, 'reg_alpha': 7.508121301806939, 'reg_lambda': 2.5414534236075124, 'scale_pos_weight': 1.9586463084723413, 'tree_method': 'hist', 'grow_policy': 'depthwise', 'max_leaves': 36, 'min_split_loss': 3.391475680156642, 'max_bin': 235}. Best is trial 31 with value: 0.407728771244436.
[I 2026-01-22 14:58:34,938] Trial 36 finished with value: 0.0 and parameters: {'max_depth': 4, 'learning_rate': 0.050455475706242224, 'n_estimators': 244, 'min_child_weight': 10, 'max_delta_step': 4, 'subsample': 0.5004048387460976, 'colsample_bytree': 0.851932622418331, 'colsample_bylevel': 0.43729581595167566, 'colsample_bynode': 0.818035

c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[I 2026-01-22 15:02:13,668] Trial 291 finished with value: 0.0 and parameters: {'max_depth': 11, 'learning_rate': 0.0639066809513193, 'n_estimators': 294, 'min_child_weight': 5, 'max_delta_step': 2, 'subsample': 0.9827520939226941, 'colsample_bytree': 0.5672037148215684, 'colsample_bylevel': 0.706311744939824, 'colsample_bynode': 0.4820650897671621, 'gamma': 2.707767201803402, 'reg_alpha': 8.445446811611799, 'reg_lambda': 9.124947506138783, 'scale_pos_weight': 1.1715455471241576, 'tree_method': 'hist', 'grow_policy': 'depthwise', 'max_leaves': 117, 'min_split_loss': 2.525229338626556, 'max_bin': 256}. Best is trial 157 with value: 0.474162679332938.
[I 2026-01-22 15:02:14,363] Trial 292 finished with value: 0.4712391269334205 and parameters: {'max_depth': 12, 'learning_rate': 0.0501900812578326, 'n_estimators': 321, 'min_child_weight': 9, 'max_delta_step': 8, 'subsample': 0.8590788582275324, 'colsample_bytree': 0.6526038473073063, 'colsample_bylevel': 0.714175221375147, 'colsample_byno

c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[I 2026-01-22 15:02:29,732] Trial 313 finished with value: 0.0 and parameters: {'max_depth': 12, 'learning_rate': 0.06735493697176936, 'n_estimators': 270, 'min_child_weight': 9, 'max_delta_step': 4, 'subsample': 0.9804930137658965, 'colsample_bytree': 0.6823326519383132, 'colsample_bylevel': 0.7051601347281032, 'colsample_bynode': 0.4813896926890546, 'gamma': 2.5561880198481046, 'reg_alpha': 7.902975796815334, 'reg_lambda': 3.2654595004453357, 'scale_pos_weight': 1.97503359056366, 'tree_method': 'hist', 'grow_policy': 'depthwise', 'max_leaves': 98, 'min_split_loss': 2.5413547020112017, 'max_bin': 254}. Best is trial 300 with value: 0.4749592347581913.
[I 2026-01-22 15:02:30,405] Trial 314 finished with value: 0.0 and parameters: {'max_depth': 7, 'learning_rate': 0.06128478230552742, 'n_estimators': 297, 'min_child_weight': 9, 'max_delta_step': 5, 'subsample': 0.987322190614858, 'colsample_bytree': 0.6181959153257958, 'colsample_bylevel': 0.6815418223493236, 'colsample_bynode': 0.51052

In [62]:
# best_params_xgb = grid_xgb.best_params_

In [63]:
best_params_xgb = study_xgb.best_params

In [64]:
print(best_params_xgb)

{'max_depth': 13, 'learning_rate': 0.08208437259509903, 'n_estimators': 263, 'min_child_weight': 9, 'max_delta_step': 8, 'subsample': 0.9877465075402022, 'colsample_bytree': 0.6653431949736452, 'colsample_bylevel': 0.7490846641284998, 'colsample_bynode': 0.4992727947138033, 'gamma': 4.243359629794118, 'reg_alpha': 8.477959607715333, 'reg_lambda': 9.473810284368634, 'scale_pos_weight': 7.039298397028539, 'tree_method': 'hist', 'grow_policy': 'depthwise', 'max_leaves': 125, 'min_split_loss': 2.306123830902162, 'max_bin': 284}


In [65]:
final_classifier = xgb.XGBClassifier(**best_params_xgb, random_state=42, eval_metric='logloss')

In [66]:
final_classifier.fit(X_train_res_retained, y_train_res)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=0.7490846641284998,
              colsample_bynode=0.4992727947138033,
              colsample_bytree=0.6653431949736452, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=4.243359629794118, grow_policy='depthwise',
              importance_type=None, interaction_constraints=None,
              learning_rate=0.08208437259509903, max_bin=284,
              max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=8,
              max_depth=13, max_leaves=125, min_child_weight=9,
              min_split_loss=2.306123830902162, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=263,
              n_jobs=None, ...)

In [67]:
y_pred_val = final_classifier.predict(X_val_retained)

In [68]:
y_proba_val = final_classifier.predict_proba(X_val_retained)[:, 1]

In [69]:
print(confusion_matrix(y_val, y_pred_val))

[[648  81]
 [ 13  24]]


In [70]:
print(accuracy_score(y_val, y_pred_val))

0.8772845953002611


In [71]:
print(precision_score(y_val, y_pred_val))

0.22857142857142856


In [72]:
print(recall_score(y_val, y_pred_val))

0.6486486486486487


In [73]:
print(fbeta_score(y_val, y_pred_val, beta=2))

0.4743083003952569


### Threshold tuning

In [74]:
best_threshold = 0.5

In [75]:
best_precision_score = 0

In [76]:
best_recall_score = 0

In [77]:
best_f2_score = 0

In [78]:
for threshold in np.arange(0.1, 0.95, 0.05):
    y_pred_val_threshold = (y_proba_val >= threshold).astype(int)
    recall = recall_score(y_val, y_pred_val_threshold)
    precision = precision_score(y_val, y_pred_val_threshold)
    f2_score = fbeta_score(y_val, y_pred_val_threshold, beta=2)
    
    print(f'Threshold {threshold} has F2 score {f2_score}')
    
    if precision >= 0.15:
        current_recall_score = recall
        if current_recall_score > best_recall_score:
            best_recall_score = current_recall_score
            best_precision_score = precision
            best_threshold = threshold
        elif current_recall_score == best_recall_score and precision > best_precision_score:
            best_precision_score = precision
            best_threshold = threshold
        
        # current_f2_score = f2_score
        # if current_f2_score > best_f2_score:
        #     best_f2_score = current_f2_score
        #     best_threshold = threshold 

Threshold 0.1 has F2 score 0.3541666666666667
Threshold 0.15000000000000002 has F2 score 0.350109409190372
Threshold 0.20000000000000004 has F2 score 0.35476718403547675
Threshold 0.25000000000000006 has F2 score 0.3669724770642202
Threshold 0.30000000000000004 has F2 score 0.3944020356234097
Threshold 0.3500000000000001 has F2 score 0.4131054131054131
Threshold 0.40000000000000013 has F2 score 0.42207792207792205
Threshold 0.45000000000000007 has F2 score 0.43795620437956206
Threshold 0.5000000000000001 has F2 score 0.4743083003952569
Threshold 0.5500000000000002 has F2 score 0.39094650205761317
Threshold 0.6000000000000002 has F2 score 0.2696078431372549
Threshold 0.6500000000000001 has F2 score 0.19553072625698323
Threshold 0.7000000000000002 has F2 score 0.09259259259259259
Threshold 0.7500000000000002 has F2 score 0.0
Threshold 0.8000000000000002 has F2 score 0.0
Threshold 0.8500000000000002 has F2 score 0.0
Threshold 0.9000000000000002 has F2 score 0.0


c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.cap

In [79]:
print(best_threshold)

0.40000000000000013


In [80]:
print(best_recall_score)

0.7027027027027027


### Retrain on training + validation set

In [81]:
X_train_full = pd.concat([X_train_res_retained, X_val_retained], axis=0, ignore_index=True)
y_train_full = pd.concat([y_train_res, y_val], axis=0, ignore_index=True)

In [82]:
final_classifier.fit(X_train_full, y_train_full)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=0.7490846641284998,
              colsample_bynode=0.4992727947138033,
              colsample_bytree=0.6653431949736452, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=4.243359629794118, grow_policy='depthwise',
              importance_type=None, interaction_constraints=None,
              learning_rate=0.08208437259509903, max_bin=284,
              max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=8,
              max_depth=13, max_leaves=125, min_child_weight=9,
              min_split_loss=2.306123830902162, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=263,
              n_jobs=None, ...)

### Evaluate on test set

In [83]:
X_test_retained = X_test[retained_columns]

In [84]:
y_pred_test = final_classifier.predict(X_test_retained)

In [85]:
y_proba_test = final_classifier.predict_proba(X_test_retained)[:, 1]

In [86]:
print(confusion_matrix(y_test, y_pred_test))

[[649  80]
 [ 14  24]]


In [87]:
print(accuracy_score(y_test, y_pred_test))

0.877444589308996


In [88]:
print(precision_score(y_test, y_pred_test))

0.23076923076923078


In [89]:
print(recall_score(y_test, y_pred_test))

0.631578947368421


In [90]:
print(fbeta_score(y_test, y_pred_test, beta=2))

0.46875


### Test with best threshold 0.45

In [91]:
y_pred_test_threshold = (y_proba_test >= best_threshold).astype(int)

In [92]:
print(confusion_matrix(y_test, y_pred_test_threshold))

[[578 151]
 [  9  29]]


In [93]:
print(accuracy_score(y_test, y_pred_test_threshold))

0.7913950456323338


In [94]:
print(precision_score(y_test, y_pred_test_threshold))

0.16111111111111112


In [95]:
print(recall_score(y_test, y_pred_test_threshold))

0.7631578947368421


In [96]:
print(fbeta_score(y_test, y_pred_test_threshold, beta=2))

0.4367469879518072
